In [2]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

data = {
    "ID": [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18],

    "Before": [
        6.42, 6.76, 6.56, 4.80, 8.43, 7.49,
        8.05, 5.05, 5.77, 3.91, 6.77, 6.44,
        6.17, 7.67, 7.34, 6.85, 5.13, 5.73
    ],

    "After4Weeks": [
        5.83, 6.20, 5.83, 4.27, 7.71, 7.12,
        7.25, 4.63, 5.31, 3.70, 6.15, 5.59,
        5.56, 7.11, 6.84, 6.40, 4.52, 5.13
    ],

    "After8Weeks": [
        5.75, 6.13, 5.71, 4.15, 7.67, 7.05,
        7.10, 4.67, 5.33, 3.66, 5.96, 5.64,
        5.51, 6.96, 6.82, 6.29, 4.45, 5.17
    ],

    "Margarine": [
        "B","B","B","A","B","A",
        "B","A","B","A","B","B",
        "A","A","A","B","A","B"
    ]
}

df = pd.DataFrame(data)

long_df = df.melt(
    id_vars=["ID", "Margarine"],
    value_vars=["Before", "After4Weeks", "After8Weeks"],
    var_name="Time",
    value_name="Cholesterol"
)

print("\nDescriptive Statistics:")
print(
    long_df.groupby(["Margarine", "Time"])["Cholesterol"]
    .agg(["mean", "std"])
)

model = smf.ols(
    "Cholesterol ~ C(Margarine) * C(Time) + C(ID)",
    data=long_df
).fit()

anova_results = anova_lm(model, typ=2)

print("\nMixed ANOVA Results:")
print(anova_results)

error_ss = anova_results.loc["Residual", "sum_sq"]

print("\nEffect Sizes (Partial Eta Squared):")

for effect in ["C(Margarine)", "C(Time)", "C(Margarine):C(Time)"]:
    effect_ss = anova_results.loc[effect, "sum_sq"]
    eta_sq = effect_ss / (effect_ss + error_ss)

    print(f"{effect}: {eta_sq:.3f}")

marg_f = anova_results.loc["C(Margarine)", "F"]
marg_p = anova_results.loc["C(Margarine)", "PR(>F)"]

time_f = anova_results.loc["C(Time)", "F"]
time_p = anova_results.loc["C(Time)", "PR(>F)"]

int_f = anova_results.loc["C(Margarine):C(Time)", "F"]
int_p = anova_results.loc["C(Margarine):C(Time)", "PR(>F)"]

print("\n" + "="*60)
print("APA STYLE REPORT")
print("="*60)

print(
    f"A two-way mixed ANOVA was conducted to examine the effect "
    f"of margarine type (A vs. B and time "
    f"(Before, After 4 Weeks, After 8 Weeks) on cholesterol levels."
)

print(
    f"\nThere was a significant main effect of margarine type, "
    f"F(1, 32) = {marg_f:.2f}, p = {marg_p:.3f}."
)

print(
    f"There was a significant main effect of time, "
    f"F(2, 32) = {time_f:.2f}, p = {time_p:.3f}."
)

print(
    f"There was a significant interaction effect between margarine "
    f"and time, F(2, 32) = {int_f:.2f}, p = {int_p:.3f}."
)

print("\nDecision:")
print("Reject the null hypothesis.")
print(
    "There is a significant difference in cholesterol levels "
    "between the two brands over time."
)


Descriptive Statistics:
                          mean       std
Margarine Time                          
A         After4Weeks  5.46875  1.387603
          After8Weeks  5.40875  1.373707
          Before       5.94500  1.428126
B         After4Weeks  6.14000  0.814589
          After8Weeks  6.07500  0.778835
          Before       6.77800  0.866472

Mixed ANOVA Results:
                         sum_sq    df           F        PR(>F)
C(Margarine)           2.710126   1.0  326.195713  2.407257e-18
C(Time)                4.319544   2.0  259.954154  1.631281e-20
C(ID)                 97.246730  17.0  688.517364  3.924308e-36
C(Margarine):C(Time)   0.079991   2.0    4.813905  1.486834e-02
Residual               0.265865  32.0         NaN           NaN

Effect Sizes (Partial Eta Squared):
C(Margarine): 0.911
C(Time): 0.942
C(Margarine):C(Time): 0.231

APA STYLE REPORT
A two-way mixed ANOVA was conducted to examine the effect of margarine type (A vs. B and time (Before, After 4 Weeks, After